<a href="https://colab.research.google.com/github/urmilapol/urmilapolprojects/blob/master/jobsearch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
# language: python
# Full corrected single-cell script (paste into one Colab cell).
# - Ensures secrets are loaded securely (Colab Secrets -> /content/.env -> getpass)
# - Adds 'inputs' dict with keys used in Task text to avoid KeyError interpolation
# - Provides DRY_RUN mode for testing without LLM/API calls
# - Does not include any hard-coded API keys; do not paste keys into notebook text

# OPTIONAL: install required packages if needed
# !pip install --quiet python-dotenv crewai langchain_groq

import os
import importlib
from datetime import datetime
from getpass import getpass

# Optional dotenv support if user uploads /content/.env
try:
    from dotenv import load_dotenv
except Exception:
    load_dotenv = None

# Detect Colab userdata API
_HAS_COLAB = False
try:
    from google.colab import userdata  # type: ignore
    _HAS_COLAB = True
except Exception:
    userdata = None

# Helper to show best-effort versions for reproducibility
def version_of(pkg):
    try:
        m = importlib.import_module(pkg)
        return getattr(m, "__version__", "unknown")
    except Exception:
        return "not installed"

print("Package versions (best-effort):", {
    "crewai": version_of("crewai"),
    "langchain_groq": version_of("langchain_groq"),
    "dotenv": version_of("dotenv"),
})

# Secret loader (Colab Secrets -> /content/.env -> interactive hidden prompt)
def load_secret(name, env_var=None):
    env_var = env_var or name
    # 1) existing environment
    val = os.getenv(env_var)
    if val:
        return val
    # 2) Colab Secrets
    if _HAS_COLAB:
        try:
            val = userdata.get(name)
            if val:
                return val
        except Exception:
            pass
    # 3) uploaded .env in /content
    if load_dotenv and os.path.exists("/content/.env"):
        load_dotenv("/content/.env")
        val = os.getenv(env_var)
        if val:
            return val
    # 4) interactive hidden prompt
    try:
        val = getpass(f"Enter {name} (hidden): ")
        if val:
            return val
    except Exception:
        pass
    return None

# Load keys (do NOT hard-code keys in the notebook)
OPENAI_KEY = load_secret("OPENAI_API_KEY")
GROQ_KEY = load_secret("GROQ_API_KEY")

if not OPENAI_KEY or not GROQ_KEY:
    raise RuntimeError(
        "Missing API key(s). Provide them via Colab Secrets (Secrets pane), upload /content/.env, "
        "or enter them at the prompt."
    )

# Export to environment so libraries that read env work
os.environ["OPENAI_API_KEY"] = OPENAI_KEY
os.environ["GROQ_API_KEY"] = GROQ_KEY

# Dry-run toggles whether to call LLM or use sample data
DRY_RUN = False

# Defaults
DEFAULT_LOCATION = "Amravati"
DEFAULT_SKILLS = "Data Science Python Spark"

def dry_run_output(location=DEFAULT_LOCATION, skills=DEFAULT_SKILLS):
    sample_result = (
        "| Company | Role | Salary | Apply Link | Experience |\n"
        "|---|---:|---:|---|---:|\n"
        "| Example Corp | Data Trainee | 3 LPA | https://example.com/apply | 0-1 yrs |\n"
        "| SampleAI | Junior Data Engineer | 4 LPA | https://sample.ai/jobs | 0-1 yrs |\n"
        "| SparkSoft | Junior ML Engineer | 5 LPA | https://sparksoft.com/apply | 0-1 yrs |\n"
        "| DataWorks | Graduate Analyst | 3.5 LPA | https://dataworks.example/apply | 0-1 yrs |\n"
        "| Infosphere | Associate Data Scientist | 6 LPA | https://infosphere.example/careers | 0-1 yrs |\n"
    )
    header = f"# MCA Jobs: {location}\n\n**Date:** (dry-run)\n\n"
    return header + sample_result

# Interactive inputs
location = input("\n🏠 City: ") or DEFAULT_LOCATION
skills = input("💻 Skills: ") or DEFAULT_SKILLS

# IMPORTANT: Provide the 'inputs' dict containing any placeholders referenced in Task strings.
# Your original Task used {url} inside expected_output; add it here (and any other placeholders).
inputs = {
    "topic": f"{skills} fresher jobs in {location}",
    # Put a default link for {url} interpolation (replace with a real link if you have one)
    "url": "https://privatecapital.mckinsey.digital/survey-templates"
}

if DRY_RUN:
    result = dry_run_output(location, skills)
else:
    # Real run: import crew and LLM, build agents/tasks and kickoff with inputs
    try:
        from crewai import Agent, Task, Crew
        from langchain_groq import ChatGroq
    except Exception as e:
        raise RuntimeError("Failed to import crewai or langchain_groq. Install required packages.") from e

    # Create LLM after keys set
    llm = ChatGroq(model="llama3-8b-8192", temperature=0)
    print("✅ LLM initialized: llama3-8b-8192")

    # Define agents (use your original backstories/roles)
    researcher = Agent(
        role="Job Researcher",
        goal=f"Find {skills} MCA fresher jobs in {location}",
        backstory="Shivaji University placement expert for IT jobs in Maharashtra",
        llm=llm,
        verbose=True,
    )

    analyzer = Agent(
        role="Job Analyzer",
        goal="Rank jobs by MCA skill match",
        backstory="Data Science professor evaluating job descriptions",
        llm=llm,
        verbose=True,
    )

    coach = Agent(
        role="Career Coach",
        goal="Create resume and interview prep materials",
        backstory="TCS placement trainer for MCA freshers",
        llm=llm,
        verbose=True,
    )

    # Tasks (ensure no stray placeholders left unmatched; we use literal f-strings for safe interpolation where appropriate)
    task1 = Task(
        description=f"""List TOP 5 {skills} fresher jobs in {location}:
1. Company + role
2. Salary 3-8 LPA
3. Apply link
4. Experience needed""",
        agent=researcher,
        expected_output="Markdown table of 5 jobs",
    )

    task2 = Task(
        description="Score 5 jobs 1-10 for MCA students. Rank top 3.",
        context=[task1],
        agent=analyzer,
        expected_output="Job ranking with scores + top 3",
    )

    # For task3 we avoid runtime-format placeholders inside the Task.expected_output;
    # if you need placeholders, ensure they exist in the `inputs` dict above.
    task3 = Task(
        description="TOP 3 jobs: resume bullet + 2 interview questions each.",
        context=[task2],
        agent=coach,
        expected_output="Interview prep + resume guide",
    )

    crew = Crew(agents=[researcher, analyzer, coach], tasks=[task1, task2, task3], verbose=2)

    # Defensive check: find placeholders used in any Task strings and confirm inputs cover them
    import re
    required_placeholders = set()
    for t in [task1, task2, task3]:
        for attr in ("description", "expected_output"):
            text = getattr(t, attr, "") or ""
            required_placeholders.update(re.findall(r"\{([A-Za-z0-9_]+)\}", text))
    missing = required_placeholders - set(inputs.keys())
    if missing:
        raise RuntimeError(f"Missing interpolation inputs required by tasks: {missing}. Add them to the 'inputs' dict before kickoff.")

    try:
        # Pass the inputs dict to kickoff so formatting placeholders are satisfied
        result = crew.kickoff(inputs=inputs)
    except Exception as exc:
        # Provide diagnostics to help debugging
        import traceback
        traceback.print_exc()
        raise RuntimeError("Crew kickoff failed. See full traceback above.") from exc

# Save output (no secrets in file)
filename = f"mca_jobs_{location.replace(' ', '_')}.md"
with open(filename, "w", encoding="utf-8") as f:
    f.write(f"# MCA Jobs: {location}\n\n**Date:** {datetime.now().strftime('%d %B %Y')}\n\n")
    f.write(result)

# Attempt to download in Colab, otherwise print path
try:
    from google.colab import files  # type: ignore
    files.download(filename)
    print("✅ Jobs downloaded (browser should start download).")
except Exception:
    print(f"Output saved to {filename} (download not available).")

print("\nDone. If you still see interpolation KeyError, add the missing placeholder names to the 'inputs' dict above.")



Package versions (best-effort): {'crewai': 'unknown', 'langchain_groq': 'unknown', 'dotenv': 'unknown'}

🏠 City: pune
💻 Skills: python


✅ LLM initialized: llama3-8b-8192
 [DEBUG]: == Working Agent: Job Researcher
 [INFO]: == Starting Task: List TOP 5 python fresher jobs in pune:
1. Company + role
2. Salary 3-8 LPA
3. Apply link
4. Experience needed


> Entering new CrewAgentExecutor chain...


Traceback (most recent call last):
  File "/tmp/ipython-input-19480/3237006868.py", line 201, in <cell line: 0>
    result = crew.kickoff(inputs=inputs)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/crewai/crew.py", line 252, in kickoff
    result = self._run_sequential_process()
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/crewai/crew.py", line 293, in _run_sequential_process
    output = task.execute(context=task_output)
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/crewai/task.py", line 173, in execute
    result = self._execute(
             ^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/crewai/task.py", line 182, in _execute
    result = agent.execute_task(
             ^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/crewai/agent.py", line 221, in execute_task
    result = self.agent_executor.invoke(
        

RuntimeError: Crew kickoff failed. See full traceback above.